# Page 5 — choosing the instrument

A screen, not a result. Eight checkpoints across two datasets and two prompt conditions, on the
same items, to answer three questions: which models blurt, whether a math-tuned small model beats
a general larger one of the same generation, and which dataset shows it best.

Page 5's numbers come from the pair this picks, measured separately in
[`page_05_how_to_reason.ipynb`](page_05_how_to_reason.ipynb). Writes `out/page_05_selection/`.

## How to run this

Roughly 640 generations across eight checkpoints, ~68 GB of downloads, a few hours on a 16 GB
GPU. Resumable: every generation is appended to `traces.jsonl` and keyed by `RUN_ID`, so an
interrupted run picks up where it stopped.

```bash
pip install "torch>=2.7" "transformers>=4.56" accelerate bitsandbytes sympy matplotlib requests
pip install reasoning-from-scratch
```

In [ ]:
# ---------------------------------------------------------------------------
# The knobs.
# ---------------------------------------------------------------------------
N_ITEMS   = 20        # paired: every model sees these same problems
FEW_SHOT_K = 2
GEN_CAP   = 1024      # bounded by the math models' 4096-token context, below
SEED      = 123

# arm: what the model is claimed to be. gen: only same-generation pairs are compared.
ROSTER = [
    dict(id="Qwen/Qwen2.5-3B-Instruct",        arm="general", gen="2.5", quant="bf16"),
    dict(id="Qwen/Qwen2.5-3B-Instruct",        arm="general", gen="2.5", quant="8bit"),   # control
    dict(id="Qwen/Qwen2.5-Coder-3B-Instruct",  arm="general", gen="2.5", quant="bf16"),
    dict(id="Qwen/Qwen2-7B-Instruct",          arm="general", gen="2",   quant="8bit"),
    dict(id="Qwen/Qwen3-1.7B",                 arm="general", gen="3",   quant="bf16", no_think=True),
    dict(id="Qwen/Qwen2-Math-1.5B-Instruct",   arm="math",    gen="2",   quant="bf16"),
    dict(id="Qwen/Qwen2.5-Math-1.5B-Instruct", arm="math",    gen="2.5", quant="bf16"),
    dict(id="Qwen/Qwen2-Math-7B-Instruct",     arm="math",    gen="2",   quant="8bit"),
    dict(id="Qwen/Qwen2.5-Math-7B-Instruct",   arm="math",    gen="2.5", quant="8bit"),
]
for m in ROSTER:
    m.setdefault("no_think", False)
    m["tag"] = m["id"].split("/")[-1] + ("" if m["quant"] == "bf16" else f"[{m['quant']}]")

# Qwen2.5-3B appears twice, bf16 and 8bit, on identical items. Without that the 7B
# results are open to "you quantised the big one"; with it the cost of 8-bit is a
# measured number rather than an argument.

print(f"{len(ROSTER)} model runs, {N_ITEMS} items, 2 datasets, 2 conditions"
      f"  ->  {len(ROSTER) * N_ITEMS * 2 * 2} generations")
for m in ROSTER:
    print(f"   {m['arm']:<8} gen {m['gen']:<4} {m['tag']}")

In [ ]:
import json
import random
import re
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch
from tqdm.auto import tqdm

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/page_05_selection"); OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"

mean, median, wilson = harness.mean, harness.median, harness.wilson
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## The two item sets

**MATH-500 level 3**, the band page 5 settled on, and **GSM8K**, which is what Wei et al.
demonstrated chain of thought on and what the page currently quotes. Both are drawn with the same
seed and hashed. Every model sees the identical problems, so accuracy differences are paired
rather than two independent samples.

Exemplars for the few-shot condition are each dataset's **own reference solutions** on held-out
items, identical across models. Page 5 harvests the model's own verified work instead; here a
fixed exemplar set is what makes the models comparable.

In [ ]:
GSM8K_URL = ("https://raw.githubusercontent.com/openai/grade-school-math/"
             "master/grade_school_math/data/test.jsonl")


def load_gsm8k(path=DATA_DIR / "gsm8k_test.jsonl"):
    """GSM8K's test split. The reference answer is the text after '#### '."""
    if not path.exists():
        path.write_text(requests.get(GSM8K_URL, timeout=60).text, encoding="utf-8")
    rows = []
    for i, line in enumerate(path.read_text(encoding="utf-8").splitlines()):
        if not line.strip():
            continue
        r = json.loads(line)
        body, _, final = r["answer"].rpartition("####")
        rows.append({"unique_id": f"gsm8k/{i}", "problem": r["question"],
                     "answer": final.strip().replace(",", ""),
                     "solution": re.sub(r"<<.*?>>", "", body).strip(),
                     "level": 0, "subject": "grade school"})
    return rows


def draw(rows, n, offset):
    rows = sorted(rows, key=lambda r: r["unique_id"])
    random.Random(SEED + offset).shuffle(rows)
    return rows[:n], rows[n:n + FEW_SHOT_K]


math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")
DATASETS = {}
for name, rows in (("math500_l3", [r for r in math500 if int(r["level"]) == 3]),
                   ("gsm8k", load_gsm8k())):
    items, shots = draw(rows, N_ITEMS, 0)
    DATASETS[name] = {"items": items, "exemplars": shots,
                      "hash": harness.sha256_of(r["unique_id"] for r in items)}
    assert not ({r["unique_id"] for r in items} & {r["unique_id"] for r in shots})
    print(f"{name:<12} {len(items)} scored + {len(shots)} exemplars   "
          f"sha256 {DATASETS[name]['hash'][:12]}   of {len(rows)} available")

print("\nan exemplar, as it will be prepended:\n")
e = DATASETS["gsm8k"]["exemplars"][0]
print(render_prompt(e["question"] if "question" in e else e["problem"]))
print(e["solution"][:300])

## Grading

Inherited from ch03 unchanged, so it is not ours to defend. The same functions serve both
datasets: GSM8K answers are integers and `extract_final_candidate` falls back to the last number,
so the only thing that changes is the reference string. The cell below checks that on both
shapes before anything is generated.

In [ ]:
CHECKS = [
    (r"The answer is \boxed{18}.",           "18",   True),    # MATH shape
    (r"She makes $18 every day.",            "18",   True),    # GSM8K shape, no box
    (r"\boxed{\frac{1}{2}}",                 "0.5",  True),
    (r"\boxed{1,024}",                       "1024", True),
    (r"I am not sure.",                      "18",   False),
    (r"Step 1: 16-3=13. Step 2: 13-4=9. So \boxed{18}.", "18", True),
]
rows = [(t, g, e, bool(grade_answer(extract_final_candidate(t), g))) for t, g, e in CHECKS]
bad = [r for r in rows if r[2] != r[3]]
if bad:
    for t, g, e, got in rows:
        print(f"{'ok ' if e == got else 'FAIL'} {extract_final_candidate(t)!r:<28} vs {g!r:<8} {got}")
    raise AssertionError(f"{len(bad)} grader check(s) behaved unexpectedly")
print(f"grader preflight: {len(rows)}/{len(rows)} as expected, both dataset shapes")

## The three conditions, and what is measured

`bare` is the harness prompt alone. `forced_blurt` adds an explicit instruction not to explain,
which is the floor. `few_shot` prepends the reference solutions.

Four numbers per cell:

- **chain share** — responses with two or more steps, under page 5's segmentation rule
- **tokens to answer** — how many tokens before the first `\boxed{`, which is how quickly it blurts
- **tokens** — total written, the cost
- **accuracy** — graded against the reference

In [ ]:
# Page 5's rule, verbatim. If it changes there it must change here, or this screen
# stops predicting what that notebook will measure.
MIN_STEP_CHARS, MIN_CHAIN_STEPS = 12, 2


def _raw_spans(text):
    for pattern in (r"\n\s*\n", r"\n", r"(?<=[.!?])\s+"):
        spans, pos = [], 0
        for m in re.finditer(pattern, text):
            if m.start() > pos:
                spans.append((pos, m.start()))
            pos = m.end()
        if pos < len(text):
            spans.append((pos, len(text)))
        spans = [(a, b) for a, b in spans if text[a:b].strip()]
        if len(spans) > 1:
            return spans
    return [(0, len(text))] if text.strip() else []


def n_steps(text):
    raw, out = _raw_spans(text), []
    for k, (a, b) in enumerate(raw):
        piece = text[a:b].strip()
        mergeable = (len(piece) < MIN_STEP_CHARS and not re.search(r"\d", piece)
                     and k != len(raw) - 1 and "\\boxed" not in piece)
        if out and mergeable:
            out[-1] = (out[-1][0], b)
        else:
            out.append((a, b))
    return len(out)


def tokens_to_answer(tokens):
    """Tokens emitted before the answer marker opens. None if it never does."""
    run = ""
    for i, t in enumerate(tokens):
        run += t
        if "\\boxed{" in run:
            return i + 1
    return None


# The floor. Asking an instruction-tuned model to stop explaining is the one lever
# a bare prompt does not pull, and it is the only condition in which any of these
# checkpoints has ever produced a one-line answer.
BLURT_SUFFIX = "\n\nGive only the final answer. No explanation, no working, no other text."


def build_prompt(condition, question, exemplars):
    stem = render_prompt(question)
    if condition == "bare":
        return stem
    if condition == "forced_blurt":
        return stem + BLURT_SUFFIX
    shots = "".join(render_prompt(e.get("question", e.get("problem"))) + "\n"
                    + e["solution"].strip() + "\n\n" for e in exemplars[:FEW_SHOT_K])
    return shots + stem


CONDITIONS = ["bare", "forced_blurt", "few_shot"]
print("conditions:", CONDITIONS, " few_shot_k =", FEW_SHOT_K)

## The screen

One model at a time, loaded, swept over both datasets and both conditions, then freed before the
next is loaded. The generation cap is clipped to whatever the checkpoint's context leaves after
the prompt: the math models carry 4,096 tokens total, not the 32,768 the general ones do, and a
few-shot prompt eats into it.

In [ ]:
RUN_ID = f"select-n{N_ITEMS}-k{FEW_SHOT_K}-cap{GEN_CAP}-{SEED}"
STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("tag", "dataset", "condition", "problem_id"))
STORE.report()

device = get_device()
DTYPE = harness.pick_dtype(device)


def sweep_model(entry):
    loaded = harness.load_model(entry["id"], device=device, dtype=DTYPE,
                                load_in_8bit=entry["quant"] == "8bit")
    model, tok = loaded["model"], loaded["tokenizer"]
    ctx = getattr(model.config, "max_position_embeddings", 32768)
    print(f"      context {ctx} tokens")

    for ds_name, ds in DATASETS.items():
        for cond in CONDITIONS:
            todo = [r for r in ds["items"]
                    if STORE.get(entry["tag"], ds_name, cond, r["unique_id"]) is None]
            bar = tqdm(ds["items"], desc=f"{entry['tag'][:22]:<22} {ds_name:<10} {cond:<9}",
                       unit="item")
            for row in bar:
                if STORE.get(entry["tag"], ds_name, cond, row["unique_id"]) is not None:
                    continue
                text = build_prompt(cond, row["problem"], ds["exemplars"])
                if entry["no_think"]:
                    text += " /no_think"     # Qwen3's in-prompt route to thinking off
                chat = tok.apply_chat_template([{"role": "user", "content": text}],
                                               tokenize=False, add_generation_prompt=True)
                n_prompt = len(tok(chat, add_special_tokens=False).input_ids)
                cap = max(64, min(GEN_CAP, ctx - n_prompt - 32))
                g = harness.complete(chat, model, tok, stop=["\nQuestion:"], max_new_tokens=cap)
                got = extract_final_candidate(g["text"])
                STORE.append({
                    "tag": entry["tag"], "model_id": entry["id"], "arm": entry["arm"],
                    "generation": entry["gen"], "quant": entry["quant"],
                    "dataset": ds_name, "condition": cond,
                    "problem_id": row["unique_id"], "question": row["problem"],
                    "ground_truth": row["answer"], "trace": g["text"], "tokens": g["tokens"],
                    "prompt_tokens": n_prompt, "cap": cap, "truncated": g["truncated"],
                    "tokens_generated": g["tokens_generated"],
                    "steps": n_steps(g["text"]),
                    "tokens_to_answer": tokens_to_answer(g["tokens"]),
                    "extracted_answer": got,
                    "correct": bool(grade_answer(got, row["answer"])),
                })
    del model, loaded
    harness.free_memory()


for entry in ROSTER:
    sweep_model(entry)
print("\nscreen complete")

## Results

In [ ]:
records = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8") if l.strip()]
records = [r for r in records if r["run_id"] == RUN_ID]


def cell(tag, ds, cond):
    rs = [r for r in records if r["tag"] == tag and r["dataset"] == ds and r["condition"] == cond]
    if not rs:
        return None
    n_correct = sum(r["correct"] for r in rs)
    tta = [r["tokens_to_answer"] for r in rs if r["tokens_to_answer"] is not None]
    return {"tag": tag, "dataset": ds, "condition": cond, "n": len(rs),
            "accuracy": n_correct / len(rs), "accuracy_ci": wilson(n_correct, len(rs)),
            "chain_share": mean([r["steps"] >= MIN_CHAIN_STEPS for r in rs]),
            "median_tokens": median([r["tokens_generated"] for r in rs]),
            "median_tokens_to_answer": median(tta) if tta else None,
            "no_answer_rate": mean([r["tokens_to_answer"] is None for r in rs]),
            "truncated": mean([r["truncated"] for r in rs])}


TABLE = [c for e in ROSTER for ds in DATASETS for cond in CONDITIONS
         if (c := cell(e["tag"], ds, cond))]

for ds in DATASETS:
    print(f"\n=== {ds} ===")
    print(f"{'model':<30}{'cond':<10}{'chain':>7}{'->answer':>10}{'tokens':>8}{'acc':>7}")
    for e in ROSTER:
        for cond in CONDITIONS:
            c = next((x for x in TABLE if x["tag"] == e["tag"] and x["dataset"] == ds
                      and x["condition"] == cond), None)
            if c is None:
                continue
            tta = "-" if c["median_tokens_to_answer"] is None else f"{c['median_tokens_to_answer']:.0f}"
            print(f"{e['tag'][:29]:<30}{cond:<10}{c['chain_share']:>6.0%}{tta:>10}"
                  f"{c['median_tokens']:>8.0f}{c['accuracy']:>7.0%}")

## The gates

Three decisions, each made by a rule rather than by eye.

**The pair.** Within a generation, does the math model beat the general one? Paired, so the test
is over problems where they disagree rather than over the two rates.

**The dataset.** It has to leave the general model room to fail, the math model room to succeed,
a gap that clears its interval, and a blurt short enough to render beside a chain.

**The quantisation control.** How much did 8-bit cost `Qwen2.5-3B` on identical items? If it is
more than a couple of points, the 7B rows are read as a lower bound rather than a measurement.

In [ ]:
BLURT_TOKENS = 60          # a blurt has to fit beside a chain on the page
GAP_MIN_PTS = 10           # smaller than this will not read at n=20


def paired(tag_a, tag_b, ds, cond="bare"):
    """Problems where exactly one of them is right. Ties carry no information."""
    a = {r["problem_id"]: r["correct"] for r in records
         if r["tag"] == tag_a and r["dataset"] == ds and r["condition"] == cond}
    b = {r["problem_id"]: r["correct"] for r in records
         if r["tag"] == tag_b and r["dataset"] == ds and r["condition"] == cond}
    both = set(a) & set(b)
    a_only = sum(1 for p in both if a[p] and not b[p])
    b_only = sum(1 for p in both if b[p] and not a[p])
    return {"n_compared": len(both), "a_only": a_only, "b_only": b_only,
            "a_rate": mean([a[p] for p in both]), "b_rate": mean([b[p] for p in both])}


PAIRS = []
for gen in sorted({e["gen"] for e in ROSTER}):
    gens = [e for e in ROSTER if e["gen"] == gen and e["arm"] == "general"]
    maths = [e for e in ROSTER if e["gen"] == gen and e["arm"] == "math"]
    for g in gens:
        for m in maths:
            for ds in DATASETS:
                p = paired(m["tag"], g["tag"], ds)
                p.update(generation=gen, math=m["tag"], general=g["tag"], dataset=ds,
                         math_wins=p["a_rate"] - p["b_rate"])
                PAIRS.append(p)

print(f"{'gen':<5}{'dataset':<12}{'math model':<30}{'vs general':<26}{'math':>6}{'gen':>6}{'delta':>7}")
for p in sorted(PAIRS, key=lambda p: -p["math_wins"]):
    print(f"{p['generation']:<5}{p['dataset']:<12}{p['math'][:29]:<30}{p['general'][:25]:<26}"
          f"{p['a_rate']:>6.0%}{p['b_rate']:>6.0%}{p['math_wins']*100:>+7.0f}")

# --- dataset gate -------------------------------------------------------
DATASET_GATE = {}
for ds in DATASETS:
    gen_cells = [c for c in TABLE if c["dataset"] == ds and c["condition"] == "bare"
                 and any(e["tag"] == c["tag"] and e["arm"] == "general" for e in ROSTER)]
    math_cells = [c for c in TABLE if c["dataset"] == ds and c["condition"] == "bare"
                  and any(e["tag"] == c["tag"] and e["arm"] == "math" for e in ROSTER)]
    best_pair = max((p for p in PAIRS if p["dataset"] == ds), key=lambda p: p["math_wins"],
                    default=None)
    blurts = [c for c in gen_cells if c["median_tokens_to_answer"] is not None
              and c["median_tokens_to_answer"] <= BLURT_TOKENS]
    DATASET_GATE[ds] = {
        "general_has_headroom": any(c["accuracy"] <= 0.85 for c in gen_cells),
        "math_off_the_floor": any(c["accuracy"] >= 0.30 for c in math_cells),
        "gap_big_enough": bool(best_pair) and best_pair["math_wins"] * 100 >= GAP_MIN_PTS,
        "a_general_model_blurts": bool(blurts),
        "blurting_models": [c["tag"] for c in blurts],
        "best_gap_pts": (best_pair["math_wins"] * 100) if best_pair else None,
    }
    g = DATASET_GATE[ds]
    g["passes"] = all(g[k] for k in ("general_has_headroom", "math_off_the_floor",
                                     "gap_big_enough", "a_general_model_blurts"))
    print(f"\n{ds}: {'PASSES' if g['passes'] else 'fails'}")
    for k in ("general_has_headroom", "math_off_the_floor", "gap_big_enough", "a_general_model_blurts"):
        print(f"   {'ok ' if g[k] else 'NO '} {k}")

# --- quantisation control ----------------------------------------------
QUANT_COST = {}
for ds in DATASETS:
    a = next((c for c in TABLE if c["tag"] == "Qwen2.5-3B-Instruct" and c["dataset"] == ds
              and c["condition"] == "bare"), None)
    b = next((c for c in TABLE if c["tag"] == "Qwen2.5-3B-Instruct[8bit]" and c["dataset"] == ds
              and c["condition"] == "bare"), None)
    if a and b:
        QUANT_COST[ds] = (a["accuracy"] - b["accuracy"]) * 100
        print(f"\n8-bit cost on {ds}: {QUANT_COST[ds]:+.0f} points "
              f"(bf16 {a['accuracy']:.0%} vs 8bit {b['accuracy']:.0%})")

In [ ]:
import matplotlib.pyplot as plt

INK = {"general": "#3b4a6b", "math": "#a2704a"}
fig, axes = plt.subplots(1, len(DATASETS), figsize=(11, 4.6), sharex=True)
for ax, ds in zip(axes, DATASETS):
    cells = [c for c in TABLE if c["dataset"] == ds and c["condition"] == "bare"]
    cells.sort(key=lambda c: c["accuracy"])
    arm = {e["tag"]: e["arm"] for e in ROSTER}
    ys = range(len(cells))
    ax.barh(list(ys), [c["accuracy"] * 100 for c in cells], height=0.6,
            color=[INK[arm[c["tag"]]] for c in cells])
    ax.errorbar([c["accuracy"] * 100 for c in cells], list(ys),
                xerr=[[(c["accuracy"] - c["accuracy_ci"][0]) * 100 for c in cells],
                      [(c["accuracy_ci"][1] - c["accuracy"]) * 100 for c in cells]],
                fmt="none", ecolor="#41474f", elinewidth=1, capsize=2)
    ax.set_yticks(list(ys))
    ax.set_yticklabels([c["tag"][:26] for c in cells], fontsize=8)
    ax.set_xlim(0, 100); ax.set_xlabel("solved (%)")
    ax.set_title(f"{ds} · bare prompt · n={N_ITEMS}", fontsize=9, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("math-tuned (brown) against general (blue), same items, 95% Wilson",
             fontsize=10, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_selection.png", dpi=160, bbox_inches="tight")
plt.show()

## The decision record

`derived.json` is what page 5 and its spec cite for "why these two checkpoints". It carries the
full grid, the paired comparisons, the gate results and the recommendation, so the choice is
re-derivable rather than asserted.

In [ ]:
passing = [ds for ds, g in DATASET_GATE.items() if g["passes"]]
best = max((p for p in PAIRS if not passing or p["dataset"] in passing),
           key=lambda p: p["math_wins"], default=None)

derived = {
    "manifest": {
        "notebook": "experiment_1_which_checkpoints_blurt.ipynb",
        "run_id": RUN_ID,
        "harness": harness.fingerprint(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "notebook_git_sha": harness.git_sha(),
        "n_items": N_ITEMS, "few_shot_k": FEW_SHOT_K, "gen_cap": GEN_CAP, "seed": SEED,
        "datasets": {k: {"hash": v["hash"], "n": len(v["items"])} for k, v in DATASETS.items()},
        "roster": ROSTER,
        "versions": {"torch": torch.__version__,
                     "transformers": __import__("transformers").__version__,
                     "device": str(device)},
    },
    "grid": TABLE,
    "paired_comparisons": PAIRS,
    "dataset_gate": DATASET_GATE,
    "quantisation_cost_pts": QUANT_COST,
    "thresholds": {"blurt_tokens": BLURT_TOKENS, "gap_min_pts": GAP_MIN_PTS,
                   "min_chain_steps": MIN_CHAIN_STEPS},
    "recommendation": None if best is None else {
        "dataset": best["dataset"],
        "math_model": best["math"],
        "general_model": best["general"],
        "generation": best["generation"],
        "math_accuracy": best["a_rate"],
        "general_accuracy": best["b_rate"],
        "gap_pts": best["math_wins"] * 100,
        "disagreements": {"math_only": best["a_only"], "general_only": best["b_only"]},
    },
}
DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"derived.json   {harness.kb(DERIVED_PATH):>9.1f} KB")
print(f"traces.jsonl   {harness.kb(TRACES_PATH):>9.1f} KB")
if derived["recommendation"]:
    r = derived["recommendation"]
    print(f"\nRECOMMENDED: {r['dataset']}, {r['math_model']} ({r['math_accuracy']:.0%}) "
          f"vs {r['general_model']} ({r['general_accuracy']:.0%}), {r['gap_pts']:+.0f} points")
else:
    print("\nNO PAIR PASSED. Read the grid above before changing the roster.")